# Data Preparation and Debiasing Protocol (`data_prep`)

## 1. Environment and Configuration (Deterministic Setup)
- Fix random seeds
- Define input and output directories
- Persist all configuration for reproducibility

In [ ]:
import os
import json
import random
import hashlib
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

try:
    import yaml
except Exception:
    yaml = None


# =========================
# Path configuration
# =========================
NOTEBOOK_DIR = Path.cwd().resolve()
THESIS_ROOT = NOTEBOOK_DIR.parent
DATASET_ROOT = THESIS_ROOT / "dataset"
ARCHIVE_ROOT = DATASET_ROOT / "archive"
TRAIN_ROOT = ARCHIVE_ROOT / "seg_train" / "seg_train"
TEST_ROOT = ARCHIVE_ROOT / "seg_test" / "seg_test"
OUT_DIR = THESIS_ROOT / "outputs" / "data_prep"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Reproducibility configuration
# =========================
CONFIG = {
    "seed": {
        "split_seed": 42,
        "shuffle_seed": 42,
        "sampler_seed": 42
    },
    "split": {
        "val_ratio": 0.10,
        "stratified": True
    },
    "dedup": {
        "enabled": True,
        "rule": "train_test_md5_conflict_keep_test_drop_train"
    },
    "preprocess": {
        "img_size": 224,
        "resize_mode": "resize_longest_side_then_pad_square",
        "pad_fill": 0,
        "normalize": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225]
        },
        "train_augment": {
            "random_horizontal_flip": 0.5,
            "color_jitter": {
                "brightness": 0.2,
                "contrast": 0.2,
                "saturation": 0.2,
                "hue": 0.02
            }
        }
    }
}


def set_seed(seed: int):
    """Set Python / NumPy randomness deterministically (to satisfy item 8)."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(CONFIG["seed"]["split_seed"])

print("[INFO] THESIS_ROOT:", THESIS_ROOT)
print("[INFO] TRAIN_ROOT :", TRAIN_ROOT)
print("[INFO] TEST_ROOT  :", TEST_ROOT)
print("[INFO] OUT_DIR    :", OUT_DIR)

## 2. Build the Data Manifest
Export `data_manifest.csv` with:
- `path`
- `split`
- `class`
- `width,height`
- `md5`

In [ ]:
def md5_file(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Compute the file MD5 for duplicate and leakage detection."""
    h = hashlib.md5()
    with file_path.open("rb") as f:
        while True:
            data = f.read(chunk_size)
            if not data:
                break
            h.update(data)
    return h.hexdigest()


def collect_split_records(split_name: str, root_dir: Path):
    """Scan one split root directory and collect image metadata."""
    records = []
    classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
    for cls in classes:
        cls_dir = root_dir / cls
        for p in cls_dir.rglob("*"):
            if not p.is_file():
                continue
            try:
                with Image.open(p) as img:
                    w, h = img.size
            except Exception:
                # Keep unreadable samples in the manifest and leave dimensions empty.
                w, h = None, None

            rel_to_dataset = p.relative_to(DATASET_ROOT).as_posix()
            records.append({
                "path": rel_to_dataset,
                "split": split_name,
                "class": cls,
                "width": w,
                "height": h,
                "md5": md5_file(p),
                "source_root": str(root_dir.relative_to(DATASET_ROOT).as_posix())
            })
    return records


train_raw = collect_split_records("train_raw", TRAIN_ROOT)
test_raw = collect_split_records("test", TEST_ROOT)
manifest_df = pd.DataFrame(train_raw + test_raw)

manifest_path = OUT_DIR / "data_manifest.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8")

print(f"[OK] Saved data manifest: {manifest_path}")
print(f"[INFO] Total manifest samples: {len(manifest_df)}")
print(manifest_df.head(3))

## 3. Duplicate and Leakage Detection (with Default Cleanup)
- Generate `duplicates_report.csv`
- Default rule: if the same MD5 appears in both train and test, keep test and drop the conflicting train samples.

In [ ]:
dup_rows = []
groups = manifest_df.groupby("md5")
for md5, g in groups:
    split_set = set(g["split"].tolist())
    if len(g) > 1:
        dup_rows.append({
            "md5": md5,
            "count": int(len(g)),
            "splits": ";".join(sorted(split_set)),
            "paths": " | ".join(g["path"].tolist())
        })

dup_df = pd.DataFrame(dup_rows).sort_values(by=["count", "md5"], ascending=[False, True]) if dup_rows else pd.DataFrame(columns=["md5", "count", "splits", "paths"])
dup_report_path = OUT_DIR / "duplicates_report.csv"
dup_df.to_csv(dup_report_path, index=False, encoding="utf-8")

# Default cleanup rule: when train_raw conflicts with test, drop train_raw.
cleaning_log = []
keep_mask = np.ones(len(manifest_df), dtype=bool)
for md5, g in groups:
    splits = set(g["split"].tolist())
    if "train_raw" in splits and "test" in splits:
        train_idxs = g[g["split"] == "train_raw"].index.tolist()
        for idx in train_idxs:
            keep_mask[idx] = False
            cleaning_log.append({
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "path": manifest_df.loc[idx, "path"],
                "md5": md5,
                "action": "drop",
                "reason": "train_test_md5_conflict_keep_test_drop_train"
            })

manifest_clean_df = manifest_df[keep_mask].copy().reset_index(drop=True)
manifest_clean_path = OUT_DIR / "data_manifest_clean.csv"
manifest_clean_df.to_csv(manifest_clean_path, index=False, encoding="utf-8")

print(f"[OK] Duplicate report: {dup_report_path}")
print(f"[OK] Cleaned manifest: {manifest_clean_path}")
print(f"[INFO] Duplicate groups detected: {len(dup_df)}")
print(f"[INFO] Samples removed: {len(cleaning_log)}")

## 4. Unified Split (train / val / test)
- Stratify the cleaned `train_raw` into `train` and `val`
- Use the official test set as `test`
- Export `split_train.txt / split_val.txt / split_test.txt / split_all.csv`

In [ ]:
def stratified_split_train_val(df_train_raw: pd.DataFrame, val_ratio: float, seed: int):
    """Perform a class-stratified train/val split."""
    rng = random.Random(seed)
    train_ids, val_ids = [], []
    for cls, g in df_train_raw.groupby("class"):
        idxs = g.index.tolist()
        rng.shuffle(idxs)
        n = len(idxs)
        n_val = int(round(n * val_ratio))
        val_ids.extend(idxs[:n_val])
        train_ids.extend(idxs[n_val:])
    rng.shuffle(train_ids)
    rng.shuffle(val_ids)
    return train_ids, val_ids


df_train_raw = manifest_clean_df[manifest_clean_df["split"] == "train_raw"].copy()
df_test = manifest_clean_df[manifest_clean_df["split"] == "test"].copy()

train_ids, val_ids = stratified_split_train_val(
    df_train_raw,
    val_ratio=CONFIG["split"]["val_ratio"],
    seed=CONFIG["seed"]["split_seed"]
)

df_train_raw.loc[:, "split"] = "_tmp"
df_train_raw.loc[train_ids, "split"] = "train"
df_train_raw.loc[val_ids, "split"] = "val"
df_train_raw = df_train_raw[df_train_raw["split"].isin(["train", "val"])]

split_all_df = pd.concat([df_train_raw, df_test], ignore_index=True)
split_all_df = split_all_df[["path", "split", "class", "width", "height", "md5", "source_root"]]
split_all_df = split_all_df.sort_values(by=["split", "class", "path"]).reset_index(drop=True)

split_all_path = OUT_DIR / "split_all.csv"
split_all_df.to_csv(split_all_path, index=False, encoding="utf-8")

def write_split_txt(df: pd.DataFrame, split_name: str, out_path: Path):
    sub = df[df["split"] == split_name].copy()
    out_path.write_text("\n".join(sub["path"].tolist()), encoding="utf-8")

split_train_txt = OUT_DIR / "split_train.txt"
split_val_txt = OUT_DIR / "split_val.txt"
split_test_txt = OUT_DIR / "split_test.txt"
write_split_txt(split_all_df, "train", split_train_txt)
write_split_txt(split_all_df, "val", split_val_txt)
write_split_txt(split_all_df, "test", split_test_txt)

print(f"[OK] Unified split file: {split_all_path}")
print(f"[OK] train: {split_train_txt} (#={len(split_all_df[split_all_df['split']=='train'])})")
print(f"[OK] val  : {split_val_txt} (#={len(split_all_df[split_all_df['split']=='val'])})")
print(f"[OK] test : {split_test_txt} (#={len(split_all_df[split_all_df['split']=='test'])})")

## 5. Class Distribution Statistics and Imbalance Ratio
Export `class_distribution.json` with:
- Class counts for each split
- Training-set imbalance ratio (max/min)

In [ ]:
distribution = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "by_split": {},
    "train_imbalance_ratio_max_min": None
}

for sp in ["train", "val", "test"]:
    cnt = split_all_df[split_all_df["split"] == sp]["class"].value_counts().sort_index()
    distribution["by_split"][sp] = {k: int(v) for k, v in cnt.to_dict().items()}

train_counts = list(distribution["by_split"]["train"].values())
if train_counts and min(train_counts) > 0:
    distribution["train_imbalance_ratio_max_min"] = float(max(train_counts) / min(train_counts))

class_dist_path = OUT_DIR / "class_distribution.json"
class_dist_path.write_text(json.dumps(distribution, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"[OK] Class distribution file: {class_dist_path}")
print(json.dumps(distribution, indent=2, ensure_ascii=False))

## 6. Versioned Preprocessing Protocol
Export `preprocess_config.yaml` (fallback to JSON if the yaml dependency is unavailable).

In [ ]:
preprocess_cfg_path_yaml = OUT_DIR / "preprocess_config.yaml"
preprocess_cfg_path_json = OUT_DIR / "preprocess_config.json"

if yaml is not None:
    with preprocess_cfg_path_yaml.open("w", encoding="utf-8") as f:
        yaml.safe_dump(CONFIG["preprocess"], f, allow_unicode=True, sort_keys=False)
    print(f"[OK] Preprocessing config saved: {preprocess_cfg_path_yaml}")
else:
    preprocess_cfg_path_json.write_text(json.dumps(CONFIG["preprocess"], indent=2, ensure_ascii=False), encoding="utf-8")
    print("[WARN] PyYAML is not installed; falling back to JSON output.")
    print(f"[OK] Preprocessing config saved: {preprocess_cfg_path_json}")

## 7. Persist Cleanup Logs and Random Seeds
- `cleaning_log.json`
- `experiment_seeds.json`

In [ ]:
cleaning_log_payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "total_removed": len(cleaning_log),
    "items": cleaning_log
}
cleaning_log_path = OUT_DIR / "cleaning_log.json"
cleaning_log_path.write_text(json.dumps(cleaning_log_payload, indent=2, ensure_ascii=False), encoding="utf-8")

seeds_payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "seed": CONFIG["seed"]
}
seed_path = OUT_DIR / "experiment_seeds.json"
seed_path.write_text(json.dumps(seeds_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"[OK] Cleanup log: {cleaning_log_path}")
print(f"[OK] Seed config: {seed_path}")

## 8. Data Integrity Verification (`verify`)
Reuse this validation function before training.
The checks include:
- Whether file paths exist
- Whether `split` values are valid
- Whether `class` is empty
- Whether train/test MD5 leakage still exists

In [ ]:
VALID_SPLITS = {"train", "val", "test"}

def verify_dataset(split_df: pd.DataFrame):
    missing_paths = []
    invalid_splits = []
    invalid_classes = []

    for i, row in split_df.iterrows():
        # `path` is relative to DATASET_ROOT.
        abs_path = DATASET_ROOT / row["path"]
        if not abs_path.exists():
            missing_paths.append(row["path"])
        if row["split"] not in VALID_SPLITS:
            invalid_splits.append({"path": row["path"], "split": row["split"]})
        if not isinstance(row["class"], str) or not row["class"].strip():
            invalid_classes.append({"path": row["path"], "class": row["class"]})

    train_md5 = set(split_df[split_df["split"] == "train"]["md5"].tolist())
    test_md5 = set(split_df[split_df["split"] == "test"]["md5"].tolist())
    leakage_md5 = sorted(list(train_md5.intersection(test_md5)))

    report = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "n_rows": int(len(split_df)),
        "missing_paths": {
            "count": int(len(missing_paths)),
            "examples": missing_paths[:20]
        },
        "invalid_splits": {
            "count": int(len(invalid_splits)),
            "examples": invalid_splits[:20]
        },
        "invalid_classes": {
            "count": int(len(invalid_classes)),
            "examples": invalid_classes[:20]
        },
        "train_test_md5_leakage": {
            "count": int(len(leakage_md5)),
            "examples": leakage_md5[:20]
        },
        "pass": bool(len(missing_paths) == 0 and len(invalid_splits) == 0 and len(invalid_classes) == 0 and len(leakage_md5) == 0)
    }
    return report


verify_report = verify_dataset(split_all_df)
verify_path = OUT_DIR / "verify_report.json"
verify_path.write_text(json.dumps(verify_report, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"[OK] Verification report: {verify_path}")
print(json.dumps(verify_report, indent=2, ensure_ascii=False))

## 9. Final Artifact Checklist
1. `data_manifest.csv`
2. `duplicates_report.csv` + default cleanup logic
3. `split_train.txt / split_val.txt / split_test.txt / split_all.csv`
4. `class_distribution.json`
5. `preprocess_config.yaml` (or JSON fallback)
6. `cleaning_log.json`
7. `verify_report.json`
8. `experiment_seeds.json`

In [ ]:
for p in sorted(OUT_DIR.glob("*")):
    print(p.name)